# 🔢 Cuaderno 01 — Álgebra Lineal

**Métodos Numéricos · Unidad 3: Sistemas de Ecuaciones Lineales**

Este cuaderno es la **puerta de entrada** a la Unidad 3. Aprenderás a manipular matrices y
vectores con una sintaxis **estilo MATLAB** que elimina la fricción de Python.

### 🎯 Objetivos
- Construir matrices y vectores con `M[[...]]`, `I[n]`, `zeros`, `ones`, `eye`.
- Hacer operaciones de renglón legibles para la **eliminación Gaussiana**.
- Formar la **matriz aumentada** con `M[[A, b]]`, al estilo de MATLAB.
- Calcular determinante, inversa y resolver sistemas con `solve`.
- Introducir **álgebra simbólica** con `var`, `jacobiano` y `mostrar`.

### 🚦 Cómo se usa este cuaderno

| Paso | Celda | Qué pasa |
|---|---|---|
| 1 | **Cargar entorno** | monta tu Drive (en Colab), trae el repositorio y carga la biblioteca y el motor de calificación |
| 2 | **Tu identificación** | lee tu correo y genera `MI_TAREA` con **tus** matrices |
| 3 | Secciones 1 a 9 | práctica libre: nada de esto se califica |
| 4 | Sección 10 | la **tarea calificable**: 11 preguntas, 2 envíos |

Las secciones 1 a 9 son para experimentar: cambia los números, rompe las celdas,
vuelve a correrlas. La sección 10 es la que cuenta, y ahí sí conviene leer el
enunciado con calma.

In [ ]:
#@title ⚙️ Cargar entorno estilo MATLAB (ejecuta PRIMERO)

import os, sys, subprocess, importlib

# --- Detecta si corremos en Google Colab ---
try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

REPO_URL = "https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales.git"


def _buscar_raiz(base, saltos=6):
    """Sube desde `base` hasta la carpeta que tiene config/config.yaml."""
    b = base
    for _ in range(saltos):
        if os.path.exists(os.path.join(b, "config", "config.yaml")):
            return os.path.abspath(b)
        padre = os.path.dirname(b)
        if padre == b:
            break
        b = padre
    return None


if ES_COLAB:
    # ============================================================
    # Google Colab: montar Drive y traer el repositorio
    # ============================================================
    # El repo se clona DENTRO de tu Drive, no en /content (que se borra al
    # cerrar la sesión). Así sobrevive entre sesiones y `git pull` solo
    # actualiza lo que cambió.
    from google.colab import drive

    drive.mount("/content/drive")

    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, "U3_Repo")
    os.makedirs(BASE, exist_ok=True)

    try:
        if not os.path.exists(os.path.join(REPO, ".git")):
            print("Clonando el repositorio (la primera vez tarda un poco)...")
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            print("Actualizando el repositorio...")
            subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
        print("✅ Repositorio listo en", REPO)
    except Exception as _exc:
        print("⚠️ No pude clonar/actualizar el repositorio:", _exc)
        print("   Si la carpeta ya existe en tu Drive, puedes continuar.")
else:
    # En tu computadora el repo ya está en disco: solo hay que encontrarlo.
    _ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code
    REPO = _buscar_raiz(os.path.dirname(os.path.abspath(_ruta_nb))) if _ruta_nb else None
    REPO = REPO or _buscar_raiz(os.getcwd())

if REPO is None:
    raise RuntimeError(
        "No encuentro la carpeta del repositorio (la que tiene config/config.yaml).\n"
        "Abre el cuaderno desde dentro de U3_Repo o revisa que el clon de Colab exista."
    )

print("Repositorio :", REPO)

# ============================================================
# Biblioteca del curso (lib/robomat.py)
# ============================================================
# En sys.path van las DOS raíces que hacen falta:
#   REPO/lib    -> robomat.py y el paquete mn_motor/
#   REPO        -> el paquete preguntas/ (las preguntas de este cuaderno)
for _ruta in (os.path.join(REPO, "lib"), REPO):
    if _ruta not in sys.path:
        sys.path.insert(0, _ruta)

import robomat as _robomat
importlib.reload(_robomat)          # recoge los cambios si editaste la libreria
from robomat import *               # noqa: F401,F403

print("✅ Entorno cargado desde:", _robomat.__file__)


# ============================================================
# Motor de calificación
# ============================================================
# En Colab SIEMPRE se usa el blob ofuscado. En tu computadora puedes escoger
# "modulos": importa lib/mn_motor/ tal cual, y así funcionan F12, el
# autocompletado y los puntos de ruptura. Es el modo para depurar.
MODO_MOTOR = "modulos"

MODO = "blob" if ES_COLAB else MODO_MOTOR

if MODO == "modulos":
    try:
        # Se borran del caché para que una edición en lib/mn_motor/ o en
        # preguntas/ se note al volver a correr la celda, sin reiniciar.
        for _n in [n for n in sys.modules
                   if n.split(".")[0] in ("mn_motor", "preguntas")]:
            del sys.modules[_n]
        # Import ESTÁTICO después del borrado: es lo que Pylance analiza.
        import mn_motor as _motor
        from mn_motor import *        # noqa: F401,F403
    except ImportError as _exc:
        # En un clon del alumno no existe preguntas/: no hay clave de
        # respuestas. Sin problema: el blob trae lo mismo.
        print("ℹ️ No pude importar el motor del repositorio:", _exc)
        print("   Uso el motor ofuscado del blob.")
        MODO = "blob"

if MODO == "blob":
    import base64 as _b64
    import zlib as _zlib

    # Cada cuaderno tiene su propio blob: aquí va el del Cuaderno 01.
    ruta_motor = os.path.join(REPO, "grader", "u3_01_algebra_lineal.txt")
    with open(ruta_motor, encoding="utf-8") as _f:
        _blob = _f.read().strip()
    _codigo = _zlib.decompress(_b64.b64decode(_blob)).decode("utf-8")
    # _REPO_U3 le dice al motor dónde está el repo: dentro del blob no existe
    # __file__, así que no puede deducirlo solo.
    os.environ["U3_REPO"] = REPO
    _MOTOR = {"__name__": "blob_u3"}
    exec(_codigo, _MOTOR)
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})


# Este cuaderno califica SU conjunto de preguntas (carpeta preguntas/u3_01_algebra_lineal).
# En modo blob ya viene incrustado, pero en modo módulos hay que declararlo.
usar_cuaderno("u3_01_algebra_lineal", "U3_01_Algebra_Lineal")

print(f"✅ Motor de calificación cargado ({MODO}) · cuaderno {cuaderno()}")
print("   Sigue con la celda de identificación.")

## 0. Tu identificación

Todo lo que sigue usa **tus** datos. Al ejecutar la celda de abajo, Colab te
pide permiso para leer tu correo institucional; de ahí se obtiene tu **Número
de Control (NC)**, y con el NC se siembran las matrices de tu tarea.

| | |
|---|---|
| 🎲 Cada alumno recibe matrices distintas | copiar la respuesta del compañero no sirve |
| 🔁 Tus datos no cambian | si vuelves mañana verás exactamente las mismas matrices |
| 📧 Hace falta tu correo `@….tecnm.mx` | con otra cuenta la celda se niega a seguir |

La tarea se genera **una sola vez** aquí; la sección 10 solo muestra las
preguntas de ese mismo `MI_TAREA`.

In [ ]:
#@title 👤 Identificación automática (correo institucional)
# ============================================================
# Uso el correo con el que montaste Drive: de ahí salen tu NC y tu nombre.
# ============================================================
if ES_COLAB:
    import requests
    import google.auth
    import google.auth.transport.requests
    from google.colab import auth

    try:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        info = requests.get(
            "https://www.googleapis.com/drive/v3/about?fields=user",
            headers={"Authorization": "Bearer " + creds.token},
            timeout=20,
        ).json()
        cuenta = info.get("user", {})
        email = cuenta.get("emailAddress", "")
        nombre = cuenta.get("displayName", "")
    except Exception as e:
        print("⚠️ No pude obtener tu correo automáticamente.")
        print("   Vuelve a ejecutar y acepta el permiso en la ventana de Google.")
        print("   Detalle:", e)
        email, nombre = "", ""
else:
    # Modo local: aquí no hay sesión de Google, escribe tu correo.
    email = "m16330887@hermosillo.tecnm.mx"      # <-- cambia por el tuyo
    nombre = ""                                 # Google solo lo da en Colab

if email and email.lower().endswith(".tecnm.mx"):
    alumno_id = email
    print("✅ Identificado:", email)
    print("   Número de control:", extraer_nc(email))
    print("   Nombre:", nombre if nombre else "(no disponible fuera de Colab)")
else:
    from IPython.display import HTML, display as _display

    alumno_id = None
    _display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;border-radius:8px;'
        'font-weight:bold">⛔ Necesito tu correo institucional (@….tecnm.mx).<br>'
        'Si iniciaste sesión con otra cuenta, ciérrala o usa una ventana de '
        'incógnito.</p>'))
    print("Correo detectado:", repr(email))

if alumno_id is None:
    raise RuntimeError("Detente: sin correo institucional no puedo generar tu tarea.")

# La tarea se genera UNA sola vez: las celdas de la sección 10 solo llaman a
# pregunta(n) sobre este MI_TAREA, así que todas hablan de lo mismo.
MI_TAREA = generar_tarea(alumno_id, nombre)
print(f"\nListo: {len(MI_TAREA['preguntas'])} preguntas para el NC {MI_TAREA['nc']}.")

## 1. Construir matrices y vectores

Un sistema $\mathbf{A}\mathbf{x}=\mathbf{b}$ se arma con una **matriz de coeficientes**
$\mathbf{A}$ y un **vector de términos independientes** $\mathbf{b}$.

### 1.1 La forma tradicional: NumPy y SymPy

Normalmente esto se hace con dos bibliotecas, según lo que necesites:

| Biblioteca | Para qué sirve | Cómo se crea una matriz |
|---|---|---|
| **NumPy** | cálculo **numérico**: los datos son números y buscas rapidez | `np.array([[1, 2], [3, 4]])` |
| **SymPy** | cálculo **simbólico**: quieres manipular letras, derivar, simplificar | `sym.Matrix([[p, q], [r, s]])` |

Cada una tiene su propia forma de crear las cosas y sus propios detalles finos.
Por ejemplo, en NumPy un vector no es ni fila ni columna hasta que se le da forma
con dobles corchetes, y mezclar un arreglo de NumPy con una matriz de SymPy
produce errores difíciles de leer:

In [ ]:
import numpy as np
import sympy as sym

A = np.array([[1, 2],
              [3, 4]])       # NumPy: matriz numérica

p, q, r, s = sym.symbols('p q r s')
As = sym.Matrix([[p, q],
                 [r, s]])    # SymPy: matriz simbólica

AxA = A * A        # NumPy: multiplica ELEMENTO A ELEMENTO  -> [[1, 4], [9, 16]]
print("A * A =\n", AxA)

AsxAs = As * As      # SymPy: PRODUCTO MATRICIAL              -> resultado distinto
print("As * As =\n", AsxAs)

AA = A @ A        # en ambas: producto matricial            -> [[7, 10], [15, 22]]
print("A @ A =\n", AA)

In [ ]:
np.linalg.solve(As, [1, 2])   # ✗ falla: NumPy exige números, no símbolos

Hay dos trampas clásicas de esas bibliotecas (la biblioteca del curso ya las
resuelve; se explica en la sección 2):

1. **`*` no significa lo mismo.** En NumPy multiplica elemento a elemento; en
   SymPy es el producto matricial. El resultado es distinto sin que haya ningún
   error que te avise.
2. **Las rutinas de NumPy no aceptan símbolos.** `np.linalg.solve` o
   `np.linalg.det` fallan con un mensaje poco claro si la matriz es simbólica.

> ✅ Regla: para el **producto matricial** usa siempre `@`. Es el único operador
> que significa lo mismo en NumPy y en SymPy.

### 1.2 La forma en este curso

La biblioteca del curso (`lib/robomat.py`) **ya carga** NumPy, SymPy y Matplotlib,
y añade una sintaxis más corta: `M[[...]]` construye la matriz y **decide sola** el
tipo, mirando si hay símbolos o solo números.

| Lo que escribes | Lo que obtienes |
|---|---|
| `M[[1, 2], [3, 4]]` | arreglo de **NumPy** (todos los valores son números) |
| `M[[p, q], [r, s]]` | matriz **simbólica** de SymPy (hay letras) |

Así el mismo código sirve para los dos mundos, sin importar nada ni recordar en
cuál de las dos bibliotecas estás.

#### La forma la deciden los corchetes (como en MATLAB)

| Escribes | Obtienes | Forma |
|---|---|---|
| `M[[1, 2, 3]]` | un **renglón** | `(1, 3)` |
| `M[[1], [2], [3]]` | una **columna** | `(3, 1)` |
| `M[[1, 2], [3, 4]]` | matriz 2×2 | `(2, 2)` |

La regla: si dentro hay **números sueltos**, es un renglón; si hay **listas**,
cada lista interior es un renglón. Es la misma idea que en MATLAB, donde
`[1 2 3]` es un renglón.

In [ ]:
# --- Forma tradicional con NumPy ---
import numpy as np

A_np = np.array([[1, 2],
                 [3, 4]])

b_np = np.array([[5], [6]])      # dos corchetes => vector COLUMNA (forma 2x1)

b_np = np.array([[5],
                 [6]])      # Más fácil de escribir y leer, pero lo mismo que arriba

print("A_np = \n", A_np)
print("tipo:", type(A_np).__name__)
print("forma de b_np:", b_np.shape, "-> (2, 1) significa columna")
print("si se escribe np.array([5, 6]) la forma es", np.array([5, 6]).shape, "-> fila (1D)")

In [ ]:
# --- Forma tradicional con SymPy ---
import sympy as sym

p, q, r, s = sym.symbols('p q r s')     # hay que declarar cada letra
A_sim = sym.Matrix([[p, q], [r, s]])

print("tipo:", type(A_sim).__name__)
mostrar(A_sim)
# mostrar() es de la biblioteca del curso para imprimir matrices bonitas

In [ ]:
# --- Forma del curso: M[[...]] decide el tipo solo ---
A_num = M[[1, 2],
          [3, 4]]

x1, x2 = var('x1', 'x2')    # declara variables simbólicas
A_simb = M[[x1, 2],
           [3, x2]]

print("M[[1, 2],\n  [3, 4]]   ->", type(A_num).__name__)
print("M[[x1, 2],\n  [3, x2]] ->", type(A_simb).__name__)
print()
print("A_num =\n", A_num)
print("\nA_simb =")
mostrar(A_simb)

# Es exactamente lo mismo que se obtendría a mano:
print("\n¿igual a np.array([[1, 2], [3, 4]])?", np.array_equal(A_num, A_np))

In [ ]:
# Constructores rápidos, con su equivalente en NumPy al lado
#
#   zeros(n) / ones(n)       -> MATRIZ n x n        (como MATLAB)
#   zeros(m, n) / ones(m, n) -> MATRIZ m x n        (como MATLAB)
#   zeros(n, 1) / zeros(1, n)-> vector columna/fila (explicito)
#   I[n]    -> identidad n x n            (cuadrada)
#   I[m, n] -> identidad rectangular m x n
#   zeros.sym / ones.sym / eye.sym      -> las mismas, pero SIMBOLICAS
#   I.sym[n] / I.sym[m, n]               -> identidad simbolica
#   var(A)                               -> convierte A a simbolica
#  (tambien existen zeros_sym(), ones_sym(), eye_sym() como forma larga)
#
# Para una diagonal desplazada (raro) usa eye(m, n, k); I[] acepta una o dos
# dimensiones a propósito, para que no se vuelva ambiguo de leer.
Z  = zeros(3)      # np.zeros((3, 3))    matriz 3x3 de ceros
Zm = zeros(3, 4)   # np.zeros((3, 4))    matriz 3x4 de ceros
O  = ones(3)       # np.ones((3, 3))     matriz 3x3 de unos
Om = ones(2, 3)    # np.ones((2, 3))     matriz 2x3 de unos
col = zeros(3, 1)  # np.zeros((3, 1))    vector COLUMNA
fila = zeros(1, 3) # np.zeros((1, 3))    vector FILA
Id = I[3]          # np.eye(3)           identidad CUADRADA 3x3
R  = I[3, 4]       # np.eye(3, 4)        identidad RECTANGULAR 3x4
E  = eye(3, 4)     # np.eye(3, 4)        forma explicita, igual que I[3, 4]

print("Z = zeros(3)      forma", Z.shape, "(matriz, no vector)")
print("O = ones(3)       forma", O.shape)
print("Zm = zeros(3, 4)  forma", Zm.shape)
print("Om = ones(2, 3)   forma", Om.shape)
print("zeros(3, 1)       forma", col.shape, "-> columna")
print("zeros(1, 3)       forma", fila.shape, "-> fila")
print("I[3]              forma", Id.shape)
print("I[3, 4]           forma", R.shape, "-> rectangular")
print("\nes lo mismo I[3, 4] que eye(3, 4)?", np.array_equal(R, E))

### 1.2.1 Constructores simbólicos

Cuando la matriz va a llevar **letras**, conviene crearla ya simbólica. Para eso
están `zeros_sym`, `ones_sym` y `eye_sym`, que son idénticas a sus versiones
numéricas pero devuelven una matriz de SymPy:

| Numérica | Simbólica (corta) | Simbólica (larga) |
|---|---|---|
| `zeros(3)` | `zeros.sym(3)` | `zeros_sym(3)` |
| `ones(2, 2)` | `ones.sym(2, 2)` | `ones_sym(2, 2)` |
| `eye(3)` | `eye.sym(3)` | `eye_sym(3)` |
| `I[3]` | `I.sym[3]` | — |

La forma corta `zeros.sym(3)` se lee como "los ceros, pero simbólicos".

La ventaja de crearlas simbólicas desde el principio es que imprime los números como si fueran símbolos en vez de flotantes con punto decimal, pero puedes ir llenándolas con números y símbolos mezclados.

Y si ya tienes una matriz numérica, `var(A)` la convierte a simbólica, igual que
`sym(A)` en MATLAB: `var` mira lo que recibe y decide sola si debe **crear
símbolos** (`var('x', 'y')`) o **convertir** (`var(A)`).

Al convertir, los números se vuelven **exactos**, sin punto decimal:

| Antes (flotante) | Después con `var()` |
|---|---|
| `1.0` | `1` |
| `0.1` | `1/10` |
| `7.85` | `157/20` |
| `0.3333333333333333` | `1/3` |
| `1e-18` | `1.00000000000000e-18` (se deja igual, para no destruirlo) |

Así el álgebra simbólica trabaja sin errores de redondeo: `1/3` es exacto,
mientras que `0.3333333333333333` no lo es.

In [ ]:
# Matriz simbolica vacia que se va llenando
A = zeros.sym(2, 2)          # forma corta; zeros_sym(2, 2) es equivalente
print("tipo:", type(A).__name__, "| forma:", A.shape)

x1, q = var('x1', 'q')
A[0, 0] = x1
A[0, 1] = 1
A[1, 0] = 3
A[1, 1] = cos(q)

print("\nA =")
mostrar(A)

print("\nones.sym(2, 3) = \n")
mostrar(ones.sym(2, 3))
print("eye.sym(3) =")
mostrar(eye.sym(3))

# Y los operadores siguen significando lo mismo
print("\nA * A  (elemento a elemento) =")
mostrar(A * A)
print("\nA @ A  (matricial) =")
mostrar(A @ A)

### 1.3 Qué te ahorra la biblioteca

| Si no la tuvieras | Con la biblioteca |
|---|---|
| `import numpy as np` y `import sympy as sym` en cada cuaderno | ya están cargadas |
| `np.array([[1, 2], [3, 4]])` | `M[[1, 2], [3, 4]]` |
| `sym.Matrix([[p, q], [r, s]])` | `M[[p, q], [r, s]]` (detecta las letras) |
| `sym.symbols('p q r s')` | `var('p', 'q', 'r', 's')` |
| `sym.init_printing()` + `display(Math(...))` para ver LaTeX | `mostrar(expresión)` |
| `np.linalg.solve(A, b)` | `solve(A, b)` |
| `np.linalg.det(A)` / `np.linalg.inv(A)` | `det(A)` / `inv(A)` |
| `np.linspace(...)` | `linspace(...)` o el atajo `lin(...)` |
| `plt.plot(...)` | `plot(...)` |

Además, `M[[...]]` evita el error más común del curso: mezclar sin querer un
arreglo numérico con una matriz simbólica. Si escribes un solo símbolo, toda la
matriz se vuelve simbólica y las operaciones siguen funcionando.

> 💡 **No es una biblioteca nueva**: por dentro es NumPy y SymPy. Todo lo que
> aprendas de ellas sigue siendo válido, y puedes usar `np.` y `sym.` cuando
> quieras.

## 2. Operaciones básicas

- Suma y resta: `A + B`, `A - B`.
- Producto matricial: `A @ B` (o `dot(A, B)`).
- Transpuesta: `A.T`.


### 2.1 Los operadores, y cuál usar

Si usaras NumPy y SymPy **por separado**, `*` significaría cosas distintas en
cada una: en NumPy multiplica elemento a elemento, pero en SymPy hace el
producto matricial.

**La biblioteca del curso lo unificó.** Las matrices que salen de `M[[...]]`
tienen la misma semántica de NumPy, sean numéricas o simbólicas:

| Escribes | Resultado | Equivale a |
|---|---|---|
| `A * B` | producto **elemento a elemento** | el `.*` de MATLAB |
| `A @ B` | producto **matricial** | el `*` de MATLAB |
| `A ** 2` | potencia **elemento a elemento** | el `^` de MATLAB, cada entrada al cuadrado |
| `A.T` | traspuesta | cambiar renglones por columnas |

Una sola regla, igual para números y para letras:

> ✅ **Regla:** `@` es producto matricial; `*` es elemento a elemento.

In [ ]:
A = M[[1, 2],
      [3, 4]]

B = M[[2, 0],
      [1, 2]]

print("A =")
print(A)
print("B =")
print(B)

print("\nA + B  (suma, no hay ambiguedad) =")
print(A + B)

print("\nA @ B  (producto MATRICIAL) =")
print(A @ B)
print("     fila 1 de A por columna 1 de B: 1*2 + 2*1 = 4")

print("\nA * B  (producto ELEMENTO A ELEMENTO) =")
print(A * B)
print("     equivale al .* de MATLAB")
print("     cada entrada por su pareja: 1*2 = 2, 2*0 = 0, 3*1 = 3, 4*2 = 8")

print("\nA ** 2  (cada entrada al cuadrado) =")
print(A ** 2)

print("\nA.T  (traspuesta; aqui coincide con NumPy y SymPy) =")
print(A.T)

In [ ]:
# Lo mismo con simbolos: el resultado NO cambia
x1, x2 = var('x1', 'x2')
S = M[[x1, 2],
      [3, x2]]
B = var(B)  # convierte B a simbolica, para que el producto sea simbólico

print("S =")
mostrar(S)

print("\nS @ B  (producto MATRICIAL) =")
mostrar(S @ B)
print("\nS * B  (ELEMENTO A ELEMENTO) =")
mostrar(S * B)
print("\nS ** 2 (cada entrada al cuadrado) =")
mostrar(S ** 2)
print("\nEs el MISMO codigo que con matrices numericas.")

### 2.2 ¿Y la traspuesta?

La traspuesta `A.T` **no sustituye** al producto elemento a elemento: son
operaciones distintas que sirven para cosas distintas.

| Operación | Para qué sirve |
|---|---|
| `A * B` (o `hadamard(A, B)`) | multiplicar dos tablas entrada por entrada (escalar columnas, calcular errores relativos, Monte Carlo) |
| `A.T` | darle la vuelta a los renglones y columnas: sirve para $\mathbf{A}^{T}\mathbf{A}$ (matrices de Gram y mínimos cuadrados), para comprobar si una matriz es **simétrica** ($\mathbf{A} = \mathbf{A}^{T}$, frecuente en los casos de la unidad) y para escribir productos punto como `x.T @ y` |

En resumen: `*` es para *combinar datos*, la traspuesta es para *cambiar
la forma* de la matriz. Usa la que corresponda al problema, no una en lugar de la
otra.

In [ ]:
A = M[[1, 2],
      [3, 4]]
B = M[[2, 0],
      [1, 2]]
print("A =")
print(A)
print("B =")
print(B)

print("\nA + B =")
print(A + B)
print("\nA @ B =")
print(A @ B)
print("\nA.T   =")
print(A.T)


## 3. Operaciones de renglón (clave para la eliminación)

La **eliminación Gaussiana** se reduce a una sola operación repetida:

$$R_i \leftarrow R_i - m_{ik}\, R_k$$

Con la sintaxis estilo MATLAB se escribe casi igual que en el pizarrón, solo que empieza desde el elemento 0 en vez de empezar en 1:


In [ ]:
A = M[[ 2,  1, -1],
      [-3, -1,  2],
      [-2,  1,  2]]

# Eliminar el elemento A[1,0] usando el pivote A[0,0]
factor = A[1, 0] / A[0, 0]
A[1, :] = A[1, :] - factor * A[0, :]
# con [1, :] se indica que se reemplaza
# todos (:) los elementos de la fila 1,
# mientras que [:, 0] indicaría
# todos los elementos de la columna 0.

print("A tras eliminar A[1,0]:\n", A)


### 3.1 Esa sintaxis NO es de nuestra biblioteca

Conviene tenerlo claro: `A[1, :]` y `A[:, 0]` son **sintaxis de NumPy y de
SymPy**, no algo que agregue nuestro `robomat.py`. Lo único que aporta la
biblioteca es la forma corta de **crear** la matriz (`M[[...]]`); de ahí en
adelante todo es Python estándar.

La buena noticia es que **funciona igual** en las matrices numéricas y en las
simbólicas:

| Escritura | Significado |
|---|---|
| `A[i, :]` | todos los elementos del **renglón** $i$ |
| `A[:, j]` | todos los elementos de la **columna** $j$ |
| `A[i, j]` | un solo elemento |
| `:` | "desde el principio hasta el final" |

### ⚠️ Trampa: matrices de enteros

Si la matriz se crea con **enteros**, la asignación en el mismo lugar **trunca**
los decimales sin avisar:

```python
A = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]])   # int64
factor = A[1, 0] / A[0, 0]        # -1.5
A[1, :] = A[1, :] - factor * A[0, :]
print(A[1, :])                     # [0 0 0]  ✗  debía ser [0, 0.5, 0.5]
```

El resultado correcto se pierde porque `A` solo puede guardar enteros. **No hay
mensaje de error**: es el peor tipo de fallo.

> ✅ Por eso usamos `M[[...]]`: siempre crea la matriz en punto flotante, así que
> la operación de renglón funciona bien. Y si quieres **exactitud total**, usa
> `var(A)` para pasar a fracciones y no perder nada.

## 4. Matriz aumentada $[\mathbf{A}\mid\mathbf{b}]$

Con `M[[A, b]]` se pega el vector $\mathbf{b}$ a la derecha de $\mathbf{A}$ y se
forma la **matriz aumentada**, igual que `[A, b]` en MATLAB. Este es el objeto que
transformarás durante la eliminación.


In [ ]:
A = M[[3, -0.1, -0.2],
      [0.1,  7, -0.3],
      [0.3, -0.2, 10]]
b = M[[7.85],
      [-19.3],
      [71.4]]

Ab = M[[A, b]]
print("Matriz aumentada [A|b] =")
print(Ab)


### 4.1 Concatenar bloques

Para unir matrices la biblioteca usa **la misma idea que MATLAB**: metes los
bloques dentro de `M[[...]]` y ella decide la dirección según cómo los acomodes.

| Escribes | Equivale a MATLAB | Resultado |
|---|---|---|
| `M[[A, b]]` | `[A, b]` | uno al lado del otro (**horizontal**) |
| `M[[A], [B]]` | `[A; B]` | uno debajo del otro (**vertical**) |
| `M[[A, B], [C, D]]` | `[A B; C D]` | matriz por **bloques** |

> ✅ **La forma la decide lo que escribes**, igual que en MATLAB:
>
> | Escribes | Obtienes |
> |---|---|
> | `M[[1, 2, 3]]` | **renglón** `(1, 3)` |
> | `M[[1], [2], [3]]` | **columna** `(3, 1)` |
> | `M[[1, 2], [3, 4]]` | matriz `(2, 2)`: cada lista interior es un renglón |
>
> La regla es simple: si dentro hay **números sueltos**, es un renglón; si dentro
> hay **listas**, cada lista es un renglón.

In [ ]:
# Dos renglones: se apilan VERTICALMENTE con M[[f1], [f2]]
f1 = M[[1, 2, 3]]        # numeros sueltos -> forma (1, 3), un renglon
f2 = M[[4, 5, 6]]
print("f1 tiene forma", f1.shape, "-> un renglon")

print("\nM[[f1],\n  [f2]] (concatenación vertical) =")
print(M[[f1],
        [f2]])

# Dos columnas: se apilan HORIZONTALMENTE con M[[c1, c2]]
c1 = M[[1],
       [2],
       [3]]    # listas -> forma (3, 1), una columna
c2 = M[[4],
       [5],
       [6]]
print("\nc1 tiene forma", c1.shape, "-> una columna")
print("\nM[[c1, c2]] (horizontal) =")
print(M[[c1, c2]])

#### Con matrices simbólicas

`M[[...]]` funciona igual con letras, así que **no hay que hacer nada especial**:

| Escribes | Resultado |
|---|---|
| `M[[S, bb]]` | matriz aumentada simbólica |
| `M[[S], [S]]` | dos bloques apilados en vertical |

Si alguna vez necesitas llamar la operación directamente, existen
`apilar_columnas` y `apilar_filas` (y sus alias de MATLAB `horzcat`, `vertcat`
y `cat(dim, ...)`).

#### ¿Y `c_` / `r_` de NumPy?

`c_` y `r_` son de NumPy y **no** son la forma del curso:

- Fallan si hay símbolos (intentan convertir todo a número).
- Y con `np.array([A, b])` ni siquiera se concatenan: se anidan.

```python
np.array([A, B])        # forma (2, 2, 2)  ✗  ¡tres dimensiones!
np.array([A, b])        # ValueError: no puede formar el arreglo
```

Con `M[[A, b]]` el resultado es la matriz aumentada de verdad, forma `(2, 3)`.

In [ ]:
x = var('x')

# La misma forma funciona con simbolos: matriz aumentada
S = var([[x, 1],
         [2, x]])
bb = var([[3],
          [4]])
print("M[[S, bb]] forma:", M[[S, bb]].shape)
mostrar(M[[S, bb]])

In [ ]:
# Y para apilar renglones
r1 = var([[x, 2, 3]])
r2 = var([[4, 5, 6]])
print("\nM[[r1],\n  [r2]] forma:", M[[r1], [r2]].shape)
mostrar(M[[r1], [r2]])

print("\nCon simbolos funciona exactamente igual: no hay que hacer nada especial.")

In [ ]:
# La forma mas comoda: meter las matrices dentro de M[[...]]
A = M[[1, 2],
      [3, 4]]
b = M[[5],
      [6]]
B = M[[7, 8],
      [9, 10]]

print("Horizontal  M[[A, b]]        forma", M[[A, b]].shape)
print(M[[A, b]])

In [ ]:
print("\nVertical\nM[[A],\n  [B]]      forma", M[[A], [B]].shape)
print(M[[A],
        [B]])

In [ ]:
print("\nBloques\nM[[A, B],\n  [A, B]] forma", M[[A, B], [A, B]].shape)
print(M[[A, B],
        [A, B]])

In [ ]:
print("\nNombres de MATLAB: horzcat(A, b) =", horzcat(A, b).shape,
      "| vertcat(A, B) =", vertcat(A, B).shape)

# Lo que NO hay que hacer: np.array([A, b]) no concatena
print("\nnp.array([A, B]) tiene forma", np.array([A, B]).shape,
      "<- 3 dimensiones, NO es una matriz")
try:
    np.array([A, b])
except Exception as e:
    print("np.array([A, b]) ni siquiera se puede construir:", type(e).__name__)

## 5. Determinante e inversa

- `det(A)`: determinante. Un determinante **cercano a cero** avisa de un sistema mal condicionado.
- `inv(A)`: matriz inversa $\mathbf{A}^{-1}$.


In [ ]:
A = M[[1, 2],
      [3, 4]]

print("det(A)   =", det(A))
print("inv(A)   =")
print(inv(A))
print("A @ inv(A) =")   # debe dar la identidad
print(A @ inv(A))

## 6. Resolver $\mathbf{A}\mathbf{x}=\mathbf{b}$ con `solve`

`solve(A, b)` es el equivalente a la **barra invertida** de MATLAB `A \ b`, que significa división por el lado izquierdo o $A^{-1}\mathbf{b}$. Debe ser en ese orden porque tanto $\mathbf{x}$ como $\mathbf{b}$ son vectores columna.


In [ ]:
A = M[[3, -0.1, -0.2],
      [0.1,  7, -0.3],
      [0.3, -0.2, 10]]
b = M[[7.85],
      [-19.3],
      [71.4]]

x = solve(A, b)
print("x =")
print(x)
print("Verificación A @ x =")   # debe dar b
print(A @ x)


## 7. ¿Qué significa «resolver»? Interpretación geométrica

Hasta aquí has resuelto sistemas con `solve`, pero no hemos dicho **qué representa
la solución**. Cada ecuación de $\mathbf{A}\mathbf{x}=\mathbf{b}$ es una
**restricción**: el conjunto de todos los puntos que la cumplen.

| Incógnitas | Cada ecuación es... | La solución es... |
|:---:|:---|:---|
| 2 | una **recta** en el plano | el **punto** donde se cruzan las rectas |
| 3 | un **plano** en el espacio | el **punto** donde se cruzan los tres planos |

Las figuras de abajo son **interactivas**: arrástralas con el ratón para girarlas.


In [ ]:
# @title
# Dos ecuaciones, dos incógnitas: cada ecuación es una RECTA
try:
    import plotly.graph_objects as go
except ImportError:                     # por si no está instalado
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=True)
    import plotly.graph_objects as go

x1 = np.linspace(-1.5, 4.5, 200)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x1, y=4 - 2 * x1, name="2x₁ + x₂ = 4",
                         line=dict(width=3, color="#1f77b4")))
fig.add_trace(go.Scatter(x=x1, y=(7 - x1) / 3, name="x₁ + 3x₂ = 7",
                         line=dict(width=3, color="#d62728")))
fig.add_trace(go.Scatter(x=[1], y=[2], mode="markers+text",
                         marker=dict(size=15, color="black"),
                         text=["solución (1, 2)"], textposition="top center",
                         name="solución"))
fig.update_layout(
    title="2 ecuaciones, 2 incógnitas → las rectas se cruzan en UN punto",
    xaxis_title="x₁", yaxis_title="x₂",
    width=720, height=540, hovermode="closest",
    xaxis=dict(range=[-1.5, 4.5], scaleanchor="y", gridcolor="#e8e8e8"),
    yaxis=dict(range=[-1.5, 4.5], gridcolor="#e8e8e8"))
fig.show()


In [ ]:
# @title
# Tres ecuaciones, tres incógnitas: cada ecuación es un PLANO.
# Usamos el mismo sistema del Cuaderno 02 / docs, cuya solución es (2, 3, -1).
u = np.linspace(0.4, 3.6, 40)
v = np.linspace(1.4, 4.6, 40)
U, V = np.meshgrid(u, v)

def _solido(color):
    return [[0, color], [1, color]]   # colorscale plana = color uniforme

fig = go.Figure()
fig.add_trace(go.Surface(x=U, y=V, z=2 * U + V - 8, opacity=0.62,
                         colorscale=_solido("#1f77b4"), showscale=False,
                         name="2x₁ + x₂ − x₃ = 8", showlegend=True))
fig.add_trace(go.Surface(x=U, y=V, z=(3 * U + V - 11) / 2, opacity=0.62,
                         colorscale=_solido("#d62728"), showscale=False,
                         name="−3x₁ − x₂ + 2x₃ = −11", showlegend=True))
fig.add_trace(go.Surface(x=U, y=V, z=(2 * U - V - 3) / 2, opacity=0.62,
                         colorscale=_solido("#2ca02c"), showscale=False,
                         name="−2x₁ + x₂ + 2x₃ = −3", showlegend=True))
fig.add_trace(go.Scatter3d(x=[2], y=[3], z=[-1], mode="markers+text",
                           marker=dict(size=7, color="black"),
                           text=["solución (2, 3, −1)"],
                           textposition="top center", name="solución"))
fig.update_layout(
    title="3 ecuaciones, 3 incógnitas → los tres planos se cruzan en UN punto",
    width=820, height=680,
    scene=dict(xaxis_title="x₁", yaxis_title="x₂", zaxis_title="x₃",
               xaxis=dict(range=[0.4, 3.6]),
               yaxis=dict(range=[1.4, 4.6]),
               zaxis=dict(range=[-5, 3])))
fig.show()


### 🔍 ¿Y en los problemas reales?

Los casos de ingeniería suelen tener **decenas, cientos o incluso miles**
de ecuaciones y de columnas (por ejemplo, el número de pixeles de una pantalla). Ahí ya no se pueden dibujar: con 4 incógnitas la
"figura" viviría en 4 dimensiones, y con 1000 no habría pantalla que la mostrara.

Por eso la representación geométrica sirve para **entender** qué significa
resolver, pero el trabajo real se hace con la matriz $\mathbf{A}$, el vector
$\mathbf{b}$ y el algoritmo. Ese es justo el tema del resto de la unidad y del
[Cuaderno 02](U3_02_Laboratorios_Metodos.ipynb).


## 8. Álgebra simbólica

Con `var()` creas variables simbólicas, y `M[[...]]` detecta automáticamente si la matriz
es simbólica. Es útil para **deducir** ecuaciones (jacobianos, equilibrios, cerchas)
antes de evaluar números.


In [ ]:
q1 = var('q1')

# Vector simbólico
f = M[[cos(q1)],
      [sin(q1)]]
mostrar(f)


In [ ]:
# y su jacobiano
J = jacobiano(f, [q1])
mostrar(J)

In [ ]:
# Evaluar el vector simbólico en 45 grados
valores = evaluar(f, [q1], [rad(45)])
print("Evaluado en 45° =\n", valores)

## 9. Ejercicios autocalificables

Cada ejercicio tiene una **celda de respuesta** que tú editas y una **celda de
verificación** que dice si está bien. La comparación la hace `verificar()`, que
**acepta cualquier forma de escribir la matriz**:

| Puedes responder con | ¿Sirve? |
|---|---|
| `M[[1, 2], [3, 4]]` | ✅ |
| `[[1, 2], [3, 4]]` (lista) | ✅ |
| `eye(4)` | ✅ |
| `A @ inv(A)` | ✅ |

No hace falta que la respuesta sea de un tipo especial: se compara el **valor**,
no el tipo.

### ✏️ Ejercicio 1 — matriz aumentada

Dado el sistema

$$\begin{aligned} 2x_1 + 3x_2 &= 7 \\ x_1 - x_2 &= 1 \end{aligned}$$

escribe la **matriz aumentada** $[\mathbf{A}\mid\mathbf{b}]$ en la variable `resp_1`.

In [ ]:
# ✏️ ESCRIBE TU RESPUESTA AQUI
resp_1 = M[[1, 1],
          [1, 1]]          # <-- cambia estos numeros (este ejemplo esta mal a proposito)

In [ ]:
# ✅ Verificacion del ejercicio 1 (no modificar)
_esperado_1 = M[[2, 3, 7],
               [1, -1, 1]]

if verificar(resp_1, _esperado_1):
    print("✅ Ejercicio 1 correcto")
else:
    print("❌ Ejercicio 1 incorrecto")
    print("   Tu respuesta tiene forma", np.shape(resp_1), "y se esperaba", _esperado_1.shape)
    print("   Recuerda: la aumentada lleva A a la izquierda y b a la derecha.")

### ✏️ Ejercicio 2 — resolver y verificar

Resuelve el sistema anterior con `solve` y guarda **el vector solución** en `resp_2`.

> 💡 Los coeficientes forman $\mathbf{A} = \begin{bmatrix}2&3\\1&-1\end{bmatrix}$ y
> $\mathbf{b} = \begin{bmatrix}7\\1\end{bmatrix}$.

In [ ]:
# ✏️ ESCRIBE TU RESPUESTA AQUI
A_2 = M[[2, 3],
        [1, -1]]
b_2 = M[[7],
        [1]]

resp_2 = b_2          # <-- reemplaza esto por la solucion

In [ ]:
# ✅ Verificacion del ejercicio 2 (no modificar)
_esperado_2 = M[[2],
               [1]]

if verificar(resp_2, _esperado_2):
    print("✅ Ejercicio 2 correcto")
else:
    print("❌ Ejercicio 2 incorrecto")

### ✏️ Ejercicio 3 — con símbolos

Con $\mathbf{A} = \begin{bmatrix} q & 1 \\ 0 & q \end{bmatrix}$, escribe
$\mathbf{A}^{2}$ en `resp_3`. La verificación acepta cualquier forma equivalente
(por ejemplo $q^2$ factorizado o desarrollado).

In [ ]:
# ✏️ ESCRIBE TU RESPUESTA AQUI
q_3 = var('q')
A_3 = M[[q_3, 1],
        [0, q_3]]

resp_3 = A_3            # <-- reemplaza esto por A_3 al cuadrado (ojo: usa @)

In [ ]:
# @title ✅ Verificacion del ejercicio 3
_q = var('q')
_esperado_3 = M[[_q**2, 2*_q],
               [0, _q**2]]

if verificar(resp_3, _esperado_3):
    print("✅ Ejercicio 3 correcto")
else:
    print("❌ Ejercicio 3 incorrecto")
    print("   Pista: A@A es el producto matricial; A*A seria elemento a elemento.")

In [ ]:
# @title 📊 Resumen de tu practica
_respuestas = [resp_1, resp_2, resp_3]
_esperados  = [_esperado_1, _esperado_2, _esperado_3]
_aciertos = revisar(_respuestas, _esperados)

print("Resultados:", ["✅" if a else "❌" for a in _aciertos])
print(f"Puntaje: {sum(_aciertos)} / {len(_aciertos)}")

## 10. Tarea calificable

Aquí está la tarea que **sí cuenta**. Son 11 preguntas hechas con **tus**
matrices: 8 de opción múltiple y 3 donde escribes la matriz o el número.

### Cómo se contesta

1. Ejecuta la celda de un bloque de preguntas.
2. Elige la opción o escribe la respuesta (`[[1,2,3],[4,5,6],[7,8,9]]`).
3. Pulsa **Guardar respuesta** en cada una. Sin ese botón la respuesta no se
   guarda, aunque ya la veas escrita en el widget.
4. Cuando termines las 11, ejecuta la celda de **Calificar** (no envía nada).
5. Si ya estás conforme, ejecuta la de **Enviar**.

### Reglas

| | |
|---|---|
| 👀 Intentos para calificar sin enviar | los que quieras |
| 📤 Envíos al profesor | **5** como máximo |
| 🎯 Vale la pena | calificar antes de enviar: el envío ya no se puede deshacer |

> 💡 `ver_preguntas()` te dice cuáles te faltan por guardar. Escríbelo en una
> celda nueva cuando quieras.

In [ ]:
#@title 🧪 Preguntas 1 a 3 · los operadores y las formas
pregunta(1)
pregunta(2)
pregunta(3)

In [ ]:
#@title 🧪 Preguntas 4 a 6 · construir matrices y trasponer
pregunta(4)
pregunta(5)
pregunta(6)

In [ ]:
#@title 🧪 Preguntas 7 y 8 · constructores y trampas
pregunta(7)
pregunta(8)

In [ ]:
#@title ✍️ Preguntas 9 a 11 · con tus matrices
pregunta(9)
pregunta(10)
pregunta(11)

In [ ]:
#@title 📝 Calificar (sin enviar) — úsala las veces que quieras
calificar()

In [ ]:
#@title 📤 Enviar la calificación (máximo 5 intentos)
enviar(alumno_id)

In [ ]:
#@title 🔎 Comprobar lo que ya está guardado (no envía nada)
consultar_calificacion()

---
### ✅ ¿Qué sigue?

Ya dominas la sintaxis básica. En el **Cuaderno 02** la usarás para programar los
métodos numéricos de la unidad: **Eliminación Gaussiana**, **Descomposición LU**,
**Gauss-Seidel** y **Thomas/Cholesky**.

📖 Consulta la teoría en [`docs/metodos/`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/tree/main/docs/metodos).

📤 Antes de cerrar: si ya contestaste la sección 10, ejecuta `calificar()` y luego
`enviar(alumno_id)`. Recuerda que solo tienes **2 envíos**.